# Stage 3 -- Theme Allocation: Combined Full Moments

## Input
- `Data/Data_Collection/Final/Stage_3_Model_Ready/themes/combined_means_theme_assignment.csv` -- the verified means theme assignment, used as the source of truth for base factor → theme mapping
- `Data/Data_Collection/Final/Stage_3_Model_Ready/model_market_combined_full_moments.parquet` -- schema read only, to obtain exact feature column names
- `Data/Data_Collection/Final/Stage_3_Model_Ready/combined_full_moments_factor_inventory.csv` -- source/description/moment_type metadata for each feature

## Purpose
Assigns every feature in the combined full moments table to the same hierarchical theme/subtheme taxonomy defined in Notebook 01. Rather than re-specifying the mapping from scratch, this notebook derives each moment column's theme by stripping its moment suffix to recover the base factor name, then looking up that base factor in the already-verified means assignment. This guarantees that all five moments of a given factor (cwmean, cwstd, cwskew, cwkurt, spread) inherit exactly the same theme and subtheme as their cwmean counterpart in the means table.

---

## Pipeline

### Step 1: Load the Verified Means Assignment
The `combined_means_theme_assignment.csv` from Notebook 01 is loaded and indexed by column name. This serves as the authoritative source of truth: every base factor already has a validated theme assignment.

### Step 2: Read Feature Column Names from Full Moments Table
The parquet schema is read without loading data. `date`, `target_daily_return`, and `target_monthly_return` are excluded.

### Step 3: Load Full Moments Inventory for Metadata
The `combined_full_moments_factor_inventory.csv` is loaded to supply `base_factor`, `moment_type`, `frequency`, `panel`, `source`, and `description` for each column.

### Step 4: Map Each Moment Column to Its Base Factor's Theme
For each feature column, two lookup strategies are tried in order:

**Try 1 -- Direct match:** if the column name exists as-is in the means assignment (macro factors, calendar features, regime indicators -- all of which appear with identical names in both tables), the theme is taken directly.

**Try 2 -- Suffix stripping:** if the column ends with one of the five moment suffixes (`_cwmean`, `_cwstd`, `_cwskew`, `_cwkurt`, `_spread`), the suffix is stripped to recover the base column name (e.g., `bid_ask_spread_cwmean` → `bid_ask_spread`, `monthly_AM_cwmean` → `monthly_AM`). The base name is then looked up in the means assignment. This works for both daily stock moment columns and monthly stock moment columns (which retain the `monthly_` prefix after suffix stripping).

Columns that match neither strategy are marked with `theme_id = -1` and `theme_name = '???'` and added to the unmatched list.

### Step 5: Validate
- **Unmatched count:** must be zero for validation to pass
- **Duplicate check:** confirms no column appears more than once
- **Theme summary table:** features and subthemes per theme, printed for review
- **Moment type breakdown:** count of cwmean, cwstd, cwskew, cwkurt, spread, and raw level columns

### Step 6: Save
Results saved to CSV.

---

## Key Design Decisions
- **Inherits from means assignment rather than re-specifying.** This is the critical design choice: rather than hand-coding theme assignments for ~2,200 moment columns, the notebook derives them automatically from the ~725-column means assignment. This guarantees consistency and eliminates the risk of a factor being assigned to different themes in the two tables.
- **Direct match handles macro/calendar/regime columns** because those appear with identical names in both the means and full moments tables (they are raw level features with no moment suffix).
- **Suffix stripping handles stock moment columns** for both daily (`bid_ask_spread_cwstd`) and monthly (`monthly_AM_cwkurt`) variants.
- **`theme_id = -1` sentinel** marks unmatched columns clearly in the output, making them easy to filter and investigate.

## Output
`Data/Data_Collection/Final/Stage_3_Model_Ready/themes/combined_full_moments_theme_assignment.csv` -- one row per feature, columns: `column`, `base_factor`, `moment_type`, `theme_id`, `theme_name`, `subtheme_id`, `subtheme_name`, `frequency`, `panel`, `source`, `description`










The 8 calendar features are correctly identified (all from Panel C Stage 1.5 Block 3 Section F)
The 5 regime indicators are correctly identified (all from Panel C Stage 1.5 Block 3 Section E)
Calendar/regime features only exist in the 4 daily/combined parquets, not the 2 monthly parquets -- the code handles this correctly by checking which columns exist before dropping
The theme IDs/names match the original assignment in document 36 exactly (Theme 3.9, 9.2, 10.4)
Regime indicators appear as raw column names in both the means and full moments theme CSVs (they're Panel C macro, not stock-level moments, so no _cwmean/_cwstd suffixes to worry about)
All 12 files from the screenshot are handled (6 parquets + 4 inventory CSVs + 2 theme CSVs)
Monthly parquets get read and re-saved unchanged (no calendar columns to find)
The validation section cross-checks everything needed

In [1]:
# %% [markdown]
# # Remove Calendar Theme & Reassign Regime Indicators
#
# This notebook:
#   1. Moves 5 regime indicator features to their parent subthemes
#   2. Drops all 8 calendar features (Theme 14.1) from datasets
#   3. Eliminates Theme 14 entirely
#   4. Saves edited copies of ALL files to Stage_4_Final_w_Calendar_Theme_Removed
#
# Regime indicator reassignments:
#   - vix_above_20, vix_above_30       → 3.9  VIX & Market Realized Volatility
#   - curve_inverted_2y10y, curve_inverted_3m10y → 9.2  Yield Curve Shape
#   - credit_stress                     → 10.4 Credit Dynamics & Volatility
#
# Calendar features DROPPED (all of subtheme 14.1):
#   day_of_week, is_monday, is_friday, month_of_year,
#   is_quarter_end, trading_days_to_month_end, is_turn_of_month, is_opex_week

# %%
import pandas as pd
import numpy as np
from pathlib import Path
import shutil

# ── Paths ────────────────────────────────────────────────────────────────────
STAGE_3 = Path('../../../Data/Data_Collection/Final/Stage_3_Model_Ready')
OUT_DIR  = Path('../../../Data/Data_Collection/Final/Stage_4_Final_w_Calendar_Theme_Removed')
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'themes').mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1: DEFINE ALL CHANGES
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STEP 1: DEFINE CHANGES")
print("=" * 90)

# Calendar features to DROP from all datasets
CALENDAR_DROP = [
    'day_of_week',
    'is_monday',
    'is_friday',
    'month_of_year',
    'is_quarter_end',
    'trading_days_to_month_end',
    'is_turn_of_month',
    'is_opex_week',
]

# Regime indicators to MOVE (not drop)
# Format: column_name → (new_theme_id, new_theme_name, new_subtheme_id, new_subtheme_name)
REGIME_MOVES = {
    'vix_above_20':        (3, 'Volatility & Options', '3.9', 'VIX & Market Realized Volatility'),
    'vix_above_30':        (3, 'Volatility & Options', '3.9', 'VIX & Market Realized Volatility'),
    'curve_inverted_2y10y':(9, 'Interest Rates & Monetary Policy', '9.2', 'Yield Curve Shape'),
    'curve_inverted_3m10y':(9, 'Interest Rates & Monetary Policy', '9.2', 'Yield Curve Shape'),
    'credit_stress':       (10, 'Credit Conditions', '10.4', 'Credit Dynamics & Volatility'),
}

# Combined: everything that was in Theme 14
ALL_THEME_14 = CALENDAR_DROP + list(REGIME_MOVES.keys())

print(f"\n  Calendar features to DROP: {len(CALENDAR_DROP)}")
for c in CALENDAR_DROP:
    print(f"    {c}")

print(f"\n  Regime indicators to MOVE: {len(REGIME_MOVES)}")
for c, (tid, tname, sid, sname) in REGIME_MOVES.items():
    print(f"    {c} → {sid} {sname}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2: EDIT PARQUET DATASETS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 2: EDIT PARQUET DATASETS")
print("=" * 90)

# Calendar and regime features live in Panel C (macro daily), so they appear
# in the daily and combined files but NOT in the monthly-only files.
#
# For daily/combined files: drop CALENDAR_DROP columns, keep regime indicators
# For monthly files: no changes needed, just copy

parquet_files = [
    'model_market_daily_means.parquet',
    'model_market_daily_full_moments.parquet',
    'model_market_monthly_means.parquet',
    'model_market_monthly_full_moments.parquet',
    'model_market_combined_means.parquet',
    'model_market_combined_full_moments.parquet',
]

for fname in parquet_files:
    src = STAGE_3 / fname
    dst = OUT_DIR / fname

    if not src.exists():
        print(f"\n  ⚠ MISSING: {src}")
        continue

    df = pd.read_parquet(src)

    # Find which calendar columns exist in this file
    cols_to_drop = [c for c in CALENDAR_DROP if c in df.columns]

    if cols_to_drop:
        n_before = df.shape[1]
        df = df.drop(columns=cols_to_drop)
        n_after = df.shape[1]
        print(f"\n  {fname}")
        print(f"    Columns: {n_before} → {n_after} (dropped {len(cols_to_drop)} calendar features)")
        for c in cols_to_drop:
            print(f"      - {c}")
    else:
        print(f"\n  {fname}")
        print(f"    No calendar features found (monthly-only file) — copying unchanged")

    # Verify regime indicators are still present (should NOT be dropped)
    regime_present = [c for c in REGIME_MOVES if c in df.columns]
    if regime_present:
        print(f"    Regime indicators retained: {len(regime_present)}")

    # Verify zero NaN
    nan_total = df.isna().sum().sum()
    assert nan_total == 0, f"FAIL: {nan_total} NaN in {fname} after editing"

    df.to_parquet(dst, index=False, engine='pyarrow')
    print(f"    ✓ Saved: {dst.name} ({df.shape[0]:,} × {df.shape[1]})")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3: EDIT FACTOR INVENTORY CSVs
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 3: EDIT FACTOR INVENTORY CSVs")
print("=" * 90)

# Factor inventories list every feature with metadata.
# We need to:
#   - Remove rows for calendar features
#   - Keep rows for regime indicators (they're just in different parquet columns now,
#     but their inventory metadata doesn't have theme info — that's in the theme CSVs)

inventory_files = [
    'daily_means_factor_inventory.csv',
    'daily_full_moments_factor_inventory.csv',
    'combined_means_factor_inventory.csv',
    'combined_full_moments_factor_inventory.csv',
]

for fname in inventory_files:
    src = STAGE_3 / fname
    dst = OUT_DIR / fname

    if not src.exists():
        print(f"\n  ⚠ MISSING: {src}")
        continue

    inv = pd.read_csv(src)
    n_before = len(inv)

    # Drop rows where the 'column' matches a calendar feature
    # For full moments files, calendar features appear as raw columns (no suffix)
    # because they come from Panel C macro (not cross-sectionally aggregated)
    inv = inv[~inv['column'].isin(CALENDAR_DROP)].reset_index(drop=True)

    n_after = len(inv)
    dropped = n_before - n_after

    print(f"\n  {fname}")
    print(f"    Rows: {n_before} → {n_after} (dropped {dropped} calendar entries)")

    # Verify regime indicators still present
    regime_in_inv = inv[inv['column'].isin(REGIME_MOVES.keys())]
    print(f"    Regime indicators retained: {len(regime_in_inv)}")

    inv.to_csv(dst, index=False)
    print(f"    ✓ Saved: {dst.name}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 4: EDIT THEME ASSIGNMENT CSVs
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 4: EDIT THEME ASSIGNMENT CSVs")
print("=" * 90)

# Theme assignment CSVs have theme_id, theme_name, subtheme_id, subtheme_name.
# We need to:
#   1. Remove rows for calendar features
#   2. Update theme/subtheme for regime indicators
#   3. Verify Theme 14 is completely eliminated

theme_files = [
    'themes/combined_means_theme_assignment.csv',
    'themes/combined_full_moments_theme_assignment.csv',
]

for fname in theme_files:
    src = STAGE_3 / fname
    dst = OUT_DIR / fname

    if not src.exists():
        print(f"\n  ⚠ MISSING: {src}")
        continue

    themes = pd.read_csv(src)
    n_before = len(themes)

    # Show what Theme 14 looks like before changes
    theme_14 = themes[themes['theme_id'] == 14]
    print(f"\n  {fname}")
    print(f"    Theme 14 rows before: {len(theme_14)}")

    # ── Step 4a: Drop calendar feature rows ──────────────────────────────
    # Calendar features appear as exact column names in means files.
    # In full moments files, they also appear as exact names (no suffix)
    # because they're Panel C macro (not stock-level moments).
    themes = themes[~themes['column'].isin(CALENDAR_DROP)].reset_index(drop=True)

    dropped_calendar = n_before - len(themes)
    print(f"    Dropped {dropped_calendar} calendar rows")

    # ── Step 4b: Update regime indicators to new themes/subthemes ────────
    moves_applied = 0
    for col_name, (new_tid, new_tname, new_sid, new_sname) in REGIME_MOVES.items():
        mask = themes['column'] == col_name
        n_matched = mask.sum()

        if n_matched > 0:
            themes.loc[mask, 'theme_id'] = new_tid
            themes.loc[mask, 'theme_name'] = new_tname
            themes.loc[mask, 'subtheme_id'] = new_sid
            themes.loc[mask, 'subtheme_name'] = new_sname
            moves_applied += n_matched
            print(f"    Moved {col_name} → {new_sid} {new_sname}")
        else:
            # In full moments file, regime indicators might not have suffixes
            # (they're raw Panel C columns), but let's check
            print(f"    ⚠ {col_name} not found in {fname}")

    print(f"    Total moves applied: {moves_applied}")

    # ── Step 4c: Verify Theme 14 is gone ─────────────────────────────────
    remaining_14 = themes[themes['theme_id'] == 14]
    if len(remaining_14) > 0:
        print(f"    ⚠ WARNING: {len(remaining_14)} rows still in Theme 14:")
        for _, row in remaining_14.iterrows():
            print(f"      {row['column']}")
    else:
        print(f"    ✓ Theme 14 completely eliminated")

    n_after = len(themes)
    print(f"    Rows: {n_before} → {n_after}")

    themes.to_csv(dst, index=False)
    print(f"    ✓ Saved: {dst.name}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 5: VALIDATION
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 5: VALIDATION")
print("=" * 90)

# ── 5a. Verify parquet files have no calendar columns ────────────────────────
print("\n  5a. Calendar columns removed from all parquets:")
for fname in parquet_files:
    path = OUT_DIR / fname
    if not path.exists():
        continue
    df = pd.read_parquet(path)
    remaining_cal = [c for c in CALENDAR_DROP if c in df.columns]
    remaining_regime = [c for c in REGIME_MOVES if c in df.columns]
    status_cal = "✓ none" if not remaining_cal else f"⚠ FOUND: {remaining_cal}"
    status_reg = f"{len(remaining_regime)} present" if remaining_regime else "n/a (monthly)"
    print(f"    {fname:<50s} calendar: {status_cal}  regime: {status_reg}")

# ── 5b. Verify theme CSVs have no Theme 14 ──────────────────────────────────
print("\n  5b. Theme 14 eliminated from theme CSVs:")
for fname in theme_files:
    path = OUT_DIR / fname
    if not path.exists():
        continue
    themes = pd.read_csv(path)
    n_14 = (themes['theme_id'] == 14).sum()
    print(f"    {fname}: Theme 14 rows = {n_14} {'✓' if n_14 == 0 else '⚠ FAIL'}")

# ── 5c. Verify regime indicators are in correct themes ───────────────────────
print("\n  5c. Regime indicator placements:")
for fname in theme_files:
    path = OUT_DIR / fname
    if not path.exists():
        continue
    themes = pd.read_csv(path)
    print(f"\n    {fname}:")
    for col_name, (expected_tid, _, expected_sid, expected_sname) in REGIME_MOVES.items():
        row = themes[themes['column'] == col_name]
        if len(row) > 0:
            actual_tid = row.iloc[0]['theme_id']
            actual_sid = row.iloc[0]['subtheme_id']
            match = (actual_tid == expected_tid) and (str(actual_sid) == str(expected_sid))
            status = "✓" if match else f"⚠ got {actual_tid}/{actual_sid}"
            print(f"      {col_name:<30s} → {expected_sid} {expected_sname}: {status}")
        else:
            print(f"      {col_name:<30s} → not found in file")

# ── 5d. Verify parquet-CSV alignment ────────────────────────────────────────
print("\n  5d. Parquet-CSV feature count alignment:")
pairs = [
    ('model_market_daily_means.parquet', 'daily_means_factor_inventory.csv'),
    ('model_market_daily_full_moments.parquet', 'daily_full_moments_factor_inventory.csv'),
    ('model_market_combined_means.parquet', 'combined_means_factor_inventory.csv'),
    ('model_market_combined_full_moments.parquet', 'combined_full_moments_factor_inventory.csv'),
]

for pq_name, csv_name in pairs:
    pq_path = OUT_DIR / pq_name
    csv_path = OUT_DIR / csv_name
    if not pq_path.exists() or not csv_path.exists():
        continue

    df = pd.read_parquet(pq_path)
    inv = pd.read_csv(csv_path)

    # Features = all columns except date and targets
    pq_features = set(c for c in df.columns
                      if c not in ['date', 'target_daily_return', 'target_monthly_return'])
    csv_features = set(inv['column'])

    in_pq_not_csv = pq_features - csv_features
    in_csv_not_pq = csv_features - pq_features

    status = "✓ aligned" if not in_pq_not_csv and not in_csv_not_pq else "⚠ MISMATCH"
    print(f"    {pq_name:<50s} pq={len(pq_features)} csv={len(csv_features)} {status}")

    if in_pq_not_csv:
        print(f"      In parquet but not CSV: {sorted(in_pq_not_csv)[:5]}")
    if in_csv_not_pq:
        print(f"      In CSV but not parquet: {sorted(in_csv_not_pq)[:5]}")

# ── 5e. Final file inventory ────────────────────────────────────────────────
print("\n  5e. Output files:")
for p in sorted(OUT_DIR.rglob('*')):
    if p.is_file():
        size = p.stat().st_size
        if size > 1e6:
            size_str = f"{size/1e6:.1f} MB"
        else:
            size_str = f"{size/1e3:.1f} KB"
        print(f"    {p.relative_to(OUT_DIR)!s:<60s} {size_str}")

# ═══════════════════════════════════════════════════════════════════════════════
# SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("COMPLETE")
print("=" * 90)

print(f"""
  Changes applied:
    DROPPED 8 calendar features from 4 daily/combined parquets + 4 inventory CSVs
    MOVED 5 regime indicators to parent subthemes in 2 theme CSVs
    ELIMINATED Theme 14 entirely

  Regime indicator new homes:
    vix_above_20, vix_above_30       → 3.9  VIX & Market Realized Volatility
    curve_inverted_2y10y/3m10y       → 9.2  Yield Curve Shape
    credit_stress                    → 10.4 Credit Dynamics & Volatility

  Files saved to: {OUT_DIR}
    6 parquet datasets (4 edited, 2 copied unchanged)
    4 factor inventory CSVs (all edited)
    2 theme assignment CSVs (all edited)
""")

STEP 1: DEFINE CHANGES

  Calendar features to DROP: 8
    day_of_week
    is_monday
    is_friday
    month_of_year
    is_quarter_end
    trading_days_to_month_end
    is_turn_of_month
    is_opex_week

  Regime indicators to MOVE: 5
    vix_above_20 → 3.9 VIX & Market Realized Volatility
    vix_above_30 → 3.9 VIX & Market Realized Volatility
    curve_inverted_2y10y → 9.2 Yield Curve Shape
    curve_inverted_3m10y → 9.2 Yield Curve Shape
    credit_stress → 10.4 Credit Dynamics & Volatility

STEP 2: EDIT PARQUET DATASETS

  model_market_daily_means.parquet
    Columns: 400 → 392 (dropped 8 calendar features)
      - day_of_week
      - is_monday
      - is_friday
      - month_of_year
      - is_quarter_end
      - trading_days_to_month_end
      - is_turn_of_month
      - is_opex_week
    Regime indicators retained: 5
    ✓ Saved: model_market_daily_means.parquet (4,299 × 392)

  model_market_daily_full_moments.parquet
    Columns: 1147 → 1139 (dropped 8 calendar features)
      -

C:\Users\Henry\AppData\Local\Temp\ipykernel_54032\3649805801.py:239: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3.9' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  themes.loc[mask, 'subtheme_id'] = new_sid
C:\Users\Henry\AppData\Local\Temp\ipykernel_54032\3649805801.py:239: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3.9' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  themes.loc[mask, 'subtheme_id'] = new_sid


    model_market_daily_means.parquet                   calendar: ✓ none  regime: 5 present
    model_market_daily_full_moments.parquet            calendar: ✓ none  regime: 5 present
    model_market_monthly_means.parquet                 calendar: ✓ none  regime: n/a (monthly)
    model_market_monthly_full_moments.parquet          calendar: ✓ none  regime: n/a (monthly)
    model_market_combined_means.parquet                calendar: ✓ none  regime: 5 present
    model_market_combined_full_moments.parquet         calendar: ✓ none  regime: 5 present

  5b. Theme 14 eliminated from theme CSVs:
    themes/combined_means_theme_assignment.csv: Theme 14 rows = 0 ✓
    themes/combined_full_moments_theme_assignment.csv: Theme 14 rows = 0 ✓

  5c. Regime indicator placements:

    themes/combined_means_theme_assignment.csv:
      vix_above_20                   → 3.9 VIX & Market Realized Volatility: ✓
      vix_above_30                   → 3.9 VIX & Market Realized Volatility: ✓
      curve_inve